In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [12]:


##########################################################
# Parameters
##########################################################

MIN_TURNS = 10
MAX_TURNS = 100

LLM_WINDOW_SIZE = 100
N_CONVERSATIONS = 200

MIN_WINDOW = 6
MAX_WINDOW = 12

HUMAN_MIN_WINDOW = 6
HUMAN_MAX_WINDOW = 8

MAX_TURN_WORDS = 200  # Cutoff threshold for individual turn length in words


In [13]:
rng = np.random.default_rng(42)

def sample_window(group):
    group = group.sort_values("message_index")

    if len(group) <= LLM_WINDOW_SIZE:
        return group

    start = rng.integers(0, len(group) - LLM_WINDOW_SIZE + 1)
    return group.iloc[start:start + LLM_WINDOW_SIZE]

In [14]:
df = pd.read_csv("Share_Chat_EN_Merged.csv")

In [15]:
df.head(5)

,conversation_id,detected_language_final,message_index,plain_text,platform,role,source,topic,turns_count,url
0,1,English,1,What common words of any length can I make fro...,chatgpt,user,chatgpt,mathematical_calculation,8,https://chatgpt.com/share/67770737-9244-8006-9...
1,1,English,2,Please provide the letters you would like me t...,chatgpt,llm,chatgpt,NaN,8,https://chatgpt.com/share/67770737-9244-8006-9...
2,1,English,3,e r t u I o c n a s d h,chatgpt,user,chatgpt,unclear,8,https://chatgpt.com/share/67770737-9244-8006-9...
3,1,English,4,Here are some common words that can be made fr...,chatgpt,llm,chatgpt,NaN,8,https://chatgpt.com/share/67770737-9244-8006-9...
4,1,English,5,And the space bar,chatgpt,user,chatgpt,greetings_and_chitchat,8,https://chatgpt.com/share/67770737-9244-8006-9...


In [16]:
set(df["topic"])

{'![img](https://api.placid.app/u/qsraj?title%5btext%5d=chatgpt%20formatting%',
 '"health_fitness_beauty_or_self_care"',
 '"ukosefu wa ujuzi hauwezi kufa, lakini uwezekano wake unaweza kuf',
 "# **unclear**\n## step 1: the conversation starts with a request to classify the user's message based on the context of the previous messages",
 "# assumptions and reasoning\nthe farmer needs to get himself and the sheep across the river. the farmer's boat can only hold one person and",
 '# capabilities\n- **greetings_and_chitchat**',
 '# capabilities\n- edit_or_critique_provided_text\n- argument_or_summary_generation\n- personal_writing_or_communication\n- write_fiction',
 '# capabilities: edit_or_critique_provided_text, other\n\n# capabilities: unclear\n\n# capabilities: unclear\n\n# capabilities: unclear\n\n# capabilities',
 '# define the conversation history\nconversation_history = ["can you relay the rest? i believe it starts with \'you are chatgpt..."]\n\n#',
 '# edit_or_critique_provided_t

In [17]:
##########################################################
# 1. Remove number-only and excessively long turns
##########################################################

numbers_only = (
    df["plain_text"]
    .fillna("")
    .str.fullmatch(r"\s*\d+\s*")
)

long_turns = (
    df["plain_text"]
    .fillna("")
    .str.split().str.len() > MAX_TURN_WORDS
)

df_clean = df.loc[~(numbers_only | long_turns)].copy()

In [18]:
##########################################################
# 2. Exclude conversations with uploads, programming, URLs, or non-target topics
##########################################################

uploaded = (
    df_clean["plain_text"]
    .fillna("")
    .str.contains(r"(?i)uploaded\s+(?:a|an)?\s*(?:file|image|photo|attachment|screenshot|pdf|document|docx|txt|csv|xlsx)", regex=True)
)

URL = (
    df_clean["plain_text"]
    .fillna("")
    .str.contains(r"<URL>|https?://\S+|www\.\S+", case=False, regex=True)
)

programming_pattern = (
    r"```|"
    r"\bdef\b|"
    r"\bclass\b|"
    r"\bimport\b|"
    r"\breturn\b|"
    r"\bfunction\b|"
    r"\bpython\b|"
    r"\bjavascript\b|"
    r"\bjava\b|"
    r"\bc\+\+\b|"
    r"\bc#\b|"
    r"\bsql\b|"
    r"\bhtml\b|"
    r"\bcss\b|"
    r"\bconsole\.log\b|"
    r"\bprint\s*\("
)

programming = (
    df_clean["plain_text"]
    .fillna("")
    .str.contains(programming_pattern,
                  case=False,
                  regex=True)
)

keep_topics = [
    "specific_info",
    "other",
    "greetings_and_chitchat",
    "relationships_and_personal_reflection"
]

non_target_topic = (
    ~df_clean["topic"].isna() &
    ~df_clean["topic"].str.lower().isin([t.lower() for t in keep_topics])
)

exclude_ids = df_clean.loc[
    uploaded | programming | URL | non_target_topic,
    "conversation_id"
].unique()

df_clean = df_clean[
    ~df_clean["conversation_id"].isin(exclude_ids)
].copy()


In [19]:
##########################################################
# 3. Recalculate conversation lengths
##########################################################

turn_counts = (
    df_clean
    .groupby("conversation_id")
    .size()
    .rename("turns")
)

df_clean = df_clean.merge(
    turn_counts,
    on="conversation_id"
)



In [20]:
##########################################################
# 4. Keep conversations with 8–100 turns
##########################################################

df_clean = df_clean[
    (df_clean["turns"] >= MIN_TURNS) &
    (df_clean["turns"] <= MAX_TURNS)
].copy()


In [21]:
# Calculate the word count for each turn in the cleaned dataset
word_counts = df_clean["plain_text"].fillna("").str.split().str.len()

# Print detailed summary statistics and quantiles
print("Turn length distribution (in words):")
print(word_counts.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]))

Turn length distribution (in words):
count    4498.000000
mean       42.696754
std        50.952315
min         1.000000
25%         8.000000
50%        18.000000
75%        63.000000
90%       129.000000
95%       161.000000
99%       193.000000
99.9%     200.000000
max       200.000000
Name: plain_text, dtype: float64


In [23]:
##########################################################
# 5b. Filter conversations by window eligibility
##########################################################

def has_valid_window(group, min_w, max_w):
    n = len(group)
    for ws in range(min_w, max_w + 1):
        for i in range(n - ws + 1):
            if group.iloc[i + ws - 1]["message_index"] - group.iloc[i]["message_index"] == ws - 1:
                return True
    return False

# Keep only conversations that have at least one valid gap-free human window (size 6-8)
eligible_ids = []
for conv_id, group in df_clean.groupby("conversation_id"):
    group_sorted = group.sort_values("message_index").reset_index(drop=True)
    if has_valid_window(group_sorted, HUMAN_MIN_WINDOW, HUMAN_MAX_WINDOW):
        eligible_ids.append(conv_id)

##########################################################
# 6. Generate Unified Dataset Pool (193 conversations)
##########################################################

# Humans and LLMs require a gap-free 6-8 turn window with context
df_unified_pool = df_clean[df_clean["conversation_id"].isin(eligible_ids)].copy()

# Using drop_duplicates to get unique conversations
mean_turns = df_unified_pool.drop_duplicates("conversation_id")["turns"].mean()
print(f"Mean turns per eligible conversation: {mean_turns:.2f}")
print(df_unified_pool.drop_duplicates("conversation_id")["turns"].min())
print(df_unified_pool.drop_duplicates("conversation_id")["turns"].max())

padded_windows = []
for conv_id, group in df_unified_pool.groupby("conversation_id"):
    group = group.sort_values("message_index").reset_index(drop=True)
    n = len(group)
    
    valid_windows = []
    for ws in range(HUMAN_MIN_WINDOW, HUMAN_MAX_WINDOW + 1):
        ws_capped = min(ws, n)
        for i in range(n - ws_capped + 1):
            if group.iloc[i + ws_capped - 1]["message_index"] - group.iloc[i]["message_index"] == ws_capped - 1:
                valid_windows.append((i, i + ws_capped))
                
    if valid_windows:
        idx = rng.integers(0, len(valid_windows))
        start_idx, end_idx = valid_windows[idx]
        
        # Calculate context indices (up to 2 before, 2 after)
        context_start = max(0, start_idx - 2)
        context_end = min(n, end_idx + 2)
        
        # Extract the full padded window
        padded_window = group.iloc[context_start:context_end].copy()
        
        # Mark should_annotate
        padded_window["should_annotate"] = False
        # The rows between start_idx and end_idx (relative to the original group) are True
        annotate_message_indices = group.iloc[start_idx:end_idx]["message_index"].values
        padded_window.loc[padded_window["message_index"].isin(annotate_message_indices), "should_annotate"] = True
        
        padded_windows.append(padded_window)

unified_df = pd.concat(padded_windows, ignore_index=True)
print(f"Total unique conversations for Unified Dataset: {unified_df['conversation_id'].nunique()}")

##########################################################
# 7. Compute Bins and Split Pilot Datasets
##########################################################

bins = [9, 11, 15, 100]
labels = ["10–11", "12–15", "16–100"]

# Calculate total original turns for the eligible conversations to assign bins
conv_turns = df_unified_pool[["conversation_id", "turns"]].drop_duplicates()
conv_turns["length_bin"] = pd.cut(
    conv_turns["turns"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Sample exactly 3 from each bin for the pilot (9 total)
pilot_conv_ids = (
    conv_turns.groupby("length_bin", observed=False, group_keys=False)
    .apply(lambda x: x.sample(n=min(3, len(x)), random_state=42))
    ["conversation_id"].tolist()
)

##########################################################
# 8. Export Unified Datasets
##########################################################

pilot_df = unified_df[unified_df["conversation_id"].isin(pilot_conv_ids)].copy()
main_df = unified_df[~unified_df["conversation_id"].isin(pilot_conv_ids)].copy()

pilot_df.to_csv("pilot_anno.csv", index=False)
main_df.to_csv("main_anno.csv", index=False)

print(f"Pilot Conversations (Unified): {pilot_df['conversation_id'].nunique()}")
print(f"Main Conversations (Unified): {main_df['conversation_id'].nunique()}")


Mean turns per eligible conversation: 13.96
10
39
Total unique conversations for Unified Dataset: 193
Pilot Conversations (Unified): 9
Main Conversations (Unified): 184


/var/folders/99/zjpq29fs355cgzb87p0s7fgw0000gn/T/ipykernel_3543/3462158296.py:86: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(3, len(x)), random_state=42))
